<a href="https://colab.research.google.com/github/XTMay/LLM_AI_Agent/blob/main/Transformer/minilm_L12_H384_psychology.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 模型介绍：Multilingual-MiniLM-L12-H384 （Microsoft）

**模型名称**：`microsoft/Multilingual-MiniLM-L12-H384`  
**模型类型**：多语种 Transformer 语言模型  
**模型规模**：12 层 Transformer（L12），隐藏层 384 维（H384）  
**参数量**：约 21M  
**发布方**：Microsoft  
**许可证**：MIT License  
**适用任务**：文本分类、语义匹配、文本嵌入生成、多语种检索、跨语言 NLP 等  

---

### 模型特点

| 特性 | 说明 |
|------|------|
| 多语种支持 | 可处理多种语言（覆盖与 XLM-R 相似的语言范围） |
| 高效轻量 | 参数量远小于 XLM-R / mBERT，但性能较好 |
| 训练方式 | 以跨语言理解任务（XNLI、MLQA 等）为参考进行预训练/评估 |
| Tokenizer | 使用 `XLMRobertaTokenizer`，模型主体为 Bert 架构 |
| 用途场景 | 嵌入生成、语义搜索、多语言分类系统、低资源部署等 |

---

### 使用时注意

1. **Tokenizer 不同于常规 BERT**  
   - 必须显式加载 `XLMRobertaTokenizer`，否则会出错  
   - 不能直接用 `AutoTokenizer.from_pretrained(...)`（官方说明）

2. 该模型为 **预训练模型**，下游任务需要微调（fine-tuning）。

3. 虽然轻量，但在非常复杂的跨语言任务中，其性能略逊于大型模型（如 XLM-R Large）。

---

### 代码示例（可直接在 Colab 运行）

```python
from transformers import AutoModel, XLMRobertaTokenizer

model_name = "microsoft/Multilingual-MiniLM-L12-H384"

tokenizer = XLMRobertaTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

text = "你好，欢迎使用多语种 MiniLM 模型！"
inputs = tokenizer(text, return_tensors="pt")
outputs = model(**inputs)

# 取 CLS token 作为句向量（示例）
sentence_embedding = outputs.last_hidden_state[:, 0, :]

sentence_embedding.shape
```


## 第1步：安装依赖

In [ ]:
# 在Colab中执行以下命令
!pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.0/488.0 kB 12.0 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.1.1
    Uninstalling sentence-transformers-5.1.1:
      Successfully uninstalled sentence-transformers-5.1.1


## 第2步：导入库

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from torch.utils.data import DataLoader
import pandas as pd
import os
import wandb
wandb.login(anonymous="allow")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: (1) Private W&B dashboard, no account required
wandb: (2) Use an existing W&B account


wandb: Enter your choice: 1、


wandb: WARNING Invalid choice


wandb: Enter your choice: 1


wandb: You chose 'Private W&B dashboard, no account required'
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: anony-mouse-391528188934707214 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 第3步：准备数据

In [ ]:
# 我们假设你有一个CSV文件，格式如下：
# sentence1,sentence2,score
# "A 58-year-old man denies his left arm belongs to him.","Right parietal cortex lesion",1
# "A 58-year-old man denies his left arm belongs to him.","Left parietal cortex lesion",0
#
# 你可以将它上传到Colab（左侧文件图标 → 上传文件）
# 或从Google Drive中读取
# 这里演示使用一个简单示例数据集

data = {
    "sentence1": [
        "A 58-year-old man denies his left arm belongs to him.",
        "A 58-year-old man denies his left arm belongs to him.",
        "Damage to the right parietal cortex causes neglect of the left side.",
        "Frontal lobe injury impairs planning and inhibition."
    ],
    "sentence2": [
        "Right parietal cortex lesion",
        "Left parietal cortex lesion",
        "Neglect of left side",
        "Planning and inhibition deficit"
    ],
    "score": [1, 0, 1, 1]
}

df = pd.DataFrame(data)
print(df.head())

                                           sentence1  \
0  A 58-year-old man denies his left arm belongs ...   
1  A 58-year-old man denies his left arm belongs ...   
2  Damage to the right parietal cortex causes neg...   
3  Frontal lobe injury impairs planning and inhib...   

                         sentence2  score  
0     Right parietal cortex lesion      1  
1      Left parietal cortex lesion      0  
2             Neglect of left side      1  
3  Planning and inhibition deficit      1  


## 第4步：构造训练样本

In [ ]:
train_examples = [
    InputExample(texts=[row['sentence1'], row['sentence2']], label=float(row['score']))
    for i, row in df.iterrows()
]

## 第5步：加载预训练模型

In [ ]:
# 我们使用心理学领域的MiniLM模型
model_name = "jtatman/minilm-L12-H384-psychology"
model = SentenceTransformer(model_name)
print(f"✅ 已加载模型：{model_name}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/205 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

✅ 已加载模型：jtatman/minilm-L12-H384-psychology


## 第6步：创建 DataLoader

In [ ]:
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=8)

## 第7步：定义损失函数

In [ ]:
# CosineSimilarityLoss 适用于语义相似任务
train_loss = losses.CosineSimilarityLoss(model)

## 第8步：训练模型

In [ ]:
# fit() 会在DataLoader上进行多轮训练
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,                # 训练轮数，可调整
    warmup_steps=100,        # 预热步数
    show_progress_bar=True   # 显示训练进度
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


## 第9步：模型评估（可选）

In [ ]:
sentences1 = [
    "Right parietal cortex damage",
    "Frontal lobe injury"
]
sentences2 = [
    "Neglect of left side",
    "Planning and inhibition deficit"
] # Test
scores = [0.9, 0.8]  # 理论相似度分数（人工标注）

evaluator = evaluation.EmbeddingSimilarityEvaluator(sentences1, sentences2, scores)
model.evaluate(evaluator)

{'pearson_cosine': -1.0, 'spearman_cosine': -0.9999999999999999}

## 第10步：保存模型

In [ ]:
output_path = "psychology-minilm-finetuned"
model.save(output_path)
print(f"✅ 模型已保存到：{output_path}")

✅ 模型已保存到：psychology-minilm-finetuned


## 第11步：测试模型效果

In [ ]:
sentences = [
    "A 58-year-old man denies his left arm belongs to him.",
    "Right parietal cortex lesion",
    "Left parietal cortex lesion"
]

embeddings = model.encode(sentences)
from sklearn.metrics.pairwise import cosine_similarity
sim = cosine_similarity(embeddings)
print("相似度矩阵：")
print(sim)

相似度矩阵：
[[ 1.0000002  -0.22942671 -0.28818053]
 [-0.22942671  0.9999998   0.99035823]
 [-0.28818053  0.99035823  1.        ]]
